# Retrieval evaluation

A demo can look good while retrieval silently regresses. We will define a tiny golden set, calculate recall@k, precision@k, and MRR, and create a simple regression gate.

## Evaluation loop

```mermaid
flowchart LR
 G[Golden set] --> R[Retriever]
 R --> M[Metrics]
 M --> C{Thresholds}
 C -->|pass| S[Continue]
 C -->|fail| I[Inspect failures]
```

In [ ]:
from examples.intermediate.evaluation import EvalCase, evaluate

cases = [
    EvalCase('rotate key', frozenset({'auth'})),
    EvalCase('health check', frozenset({'health'})),
    EvalCase('invoice', frozenset({'billing'})),
]
retrievals = {'rotate key': ['auth', 'health'], 'health check': ['health', 'auth'], 'invoice': ['billing', 'auth']}
metrics = evaluate(retrievals, cases, k=2)
metrics

In [ ]:
assert metrics['recall@k'] == 1.0
assert metrics['mrr'] == 1.0
minimum_recall = 0.9
if metrics['recall@k'] < minimum_recall:
    raise AssertionError('retrieval regression')
print('regression gate passed')

## Interpret the result

Recall asks whether relevant evidence appeared in the top-k. Precision asks how much of top-k was relevant. MRR emphasizes the rank of the first relevant result. Add answer-level metrics separately: faithfulness, citation coverage, abstention quality, latency, and cost.

## Exercise

Add a no-answer case and a permission-boundary case. Make a deliberately bad ranking fail a threshold, then inspect which query caused the regression.